In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers import AutoTokenizer
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import json

class AFQMC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt') as f:
            for idx, line in enumerate(f):
                sample = json.loads(line.strip())
                Data[idx] = sample
        return Data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    
train_data = AFQMC('data/train.json')
valid_data = AFQMC('data/dev.json')

print(train_data[0])


c:\Users\jd\.conda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'sentence1': '蚂蚁借呗等额还款可以换成先息后本吗', 'sentence2': '借呗有先息到期还本吗', 'label': '0'}


In [3]:
checkpoint = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def collote_fn(batch_samples):
    batch_setence_1, batch_setence_2 = [], []
    batch_label = []
    for sample in batch_samples:
        batch_setence_1.append(sample['sentence1'])
        batch_setence_2.append(sample['sentence2'])
        batch_label.append(int(sample['label']))
    X = tokenizer(
        batch_setence_1,
        batch_setence_2,
        padding=True,
        truncation=True, #截断
        return_tensors='pt'
    )

    y = torch.tensor(batch_label)
    return X, y

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collote_fn)
valid_dataloader= DataLoader(valid_data, batch_size=4, shuffle=False, collate_fn=collote_fn)

batch_X, batch_y = next(iter(train_dataloader))
print('batch_X shape:', {k: v.shape for k, v in batch_X.items()}) #字典推导式，打印一个batch的数据的尺寸
print('batch_y.shape:', batch_y.shape)
print(batch_X)
print(batch_y)

batch_X shape: {'input_ids': torch.Size([4, 33]), 'token_type_ids': torch.Size([4, 33]), 'attention_mask': torch.Size([4, 33])}
batch_y.shape: torch.Size([4])
{'input_ids': tensor([[ 101, 2582,  720, 2828,  955, 1446, 1068, 7308,  749,  102, 2769, 2682,
         2515, 2419, 1068,  749,  955, 1446, 8024,  679, 5543, 1086, 2458, 1423,
          886, 4500,  102,    0,    0,    0,    0,    0,    0],
        [ 101, 5709, 1446, 1068, 7308, 1400, 8024, 6206, 2582,  720, 2798, 1377,
          809, 2458, 6858,  102, 1068, 7308, 5709, 1446,  809, 1400, 6820, 1377,
          809, 1086, 2458, 6858, 1408,  102,    0,    0,    0],
        [ 101, 5698, 7937, 1146, 6809, 1168, 1914, 2208, 5543, 2458, 6858, 5709,
         1446,  102, 4500, 1914, 2208, 4916, 1146, 1377,  809, 2458, 6858, 5709,
         1446,  102,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 5709, 1446, 1059, 7583, 5632, 1220, 6820, 3621, 3300, 1164, 2622,
         1408,  102, 5709, 1446, 6392, 5390, 5632, 1220, 6820, 3621,

collate_fn的核心作用就是将一小批从数据集中取出的样本，处理成一个结构化的符合模型训练要求的张量批次。

In [ ]:
device = 'cuda' if torch.cuda.is_available else 'cpu'
print(f'Using {device} device')

class BertForPaiewiseCLS(nn.Module):
    def __init__(self):
        super(BertForPaiewiseCLS, self).__init__()
        self.Bert_encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)

    def forward(self, x):
        bert_output = self.Bert_encoder(**x) # **x 字典解包语法糖，等价于self.Bert_encoder(input_ids=tensor_a, attention_mask=tensor_b)
        cls_vectors = bert_output.last_hidden_state[:, 0, :] 
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits
    
model = BertForPaiewiseCLS().to(device)
print(model)

#### bert-base-chinese模型输出
last_hidden_state (最后一层的隐藏状态)，包含了序列中每一个 token 的最终表示向量的张量。形状：(batch_size, sequence_length, hidden_size)

last_hidden_state[:, 0, :]：取出每个句子的第一行，即[CLS]标记，是每个句子的向量化表示


In [ ]:
from torch import nn
from transformers import AutoConfig
from transformers import BertPreTrainedModel, BertModel

device = 'cuda' if torch.cuda.is_available else 'cpu'
print(f'Using {device} device')

class BertForPaiewiseCLS(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, 2)
        self.post_init()

    def forward(self, x):
        bert_output = self.bert(**x)
        cls_vectors = bert_output.last_hidden_state[:, 0, :]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits
    
config = AutoConfig.from_pretrained(checkpoint)
model = BertForPaiewiseCLS.from_pretrained(checkpoint, config=config).to(device)
print(model)

直接继承BertPreTrainedModel，会有更多实用的方法

In [ ]:
batch_X.to(device)
outputs = model(batch_X)
print(outputs.shape)

In [ ]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, loss_fn, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss:{0:>7f}')
    finish_step_num = (epoch-1)*len(dataloader) # 已完成的训练步数

    model.train()

    for step, (X, y) in enumerate(dataloader, start=1): # 索引从1开始
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_step_num + step):>7f}')
        progress_bar.update(1)

        return total_loss
    
def test_loop(dataloader, madel, mode='Test'):
    assert mode in ['Valid', 'Test']
    size = len(dataloader.dataset)
    correct = 0

    model.eval()
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            # 沿第一维查看最大值的索引，与label对比，正确返回1，错误返回0
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() 

    correct /= size # 正确率
    print(f"{mode} Accuracy: {(100*correct):>0.1f}%\n")


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
from transformers import get_scheduler

epochs = 3
num_training_steps = epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

print(num_training_steps)

In [ ]:
learning_rate = 1e-5
epoch_num = 3

loss_fn = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(train_dataloader)
)

total_loss = 0
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, loss_fn, optimizer, lr_scheduler, t+1, total_loss)
    test_loop(valid_dataloader, model, mode='Valid')
print("Done!")
